**Task 2:- Restaurant Recommendation**


*   Objective:


*   Creating a restaurant recommendation
 system based on user preferences.






In [3]:
import pandas as pd
import numpy as np

**Loading our dataset**

In [4]:
dataset = pd.read_csv("Dataset.csv")
dataset.head(2)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591


**From here our Data Preprossing will start.**

In [5]:
#Finding missing values.
clear = pd.concat([dataset.isnull().sum()], axis=1, keys = ["values"])
clear

,values
Restaurant ID,0
Restaurant Name,0
Country Code,0
City,0
Address,0
Locality,0
Locality Verbose,0
Longitude,0
Latitude,0
Cuisines,9


**Now we will handle the missing values**

In [6]:
dataset['Cuisines'] = dataset['Cuisines'].fillna(dataset['Cuisines'].mode()[0])
dataset['City'] = dataset['City'].fillna(dataset['City'].mode()[0])
# The mode is the most frequently occurring value in a dataset. We use it for filling missing values in categorical columns.

dataset['Average Cost for two'] = dataset['Average Cost for two'].fillna(dataset['Average Cost for two'].median())
dataset['Aggregate rating'] = dataset['Aggregate rating'].fillna(dataset['Aggregate rating'].median())
# The median is the middle value when all numbers are sorted in ascending order.
# Basically using the median helps to avoid the influence of these outliers.

*Here our data cleaning process got completed..*

In [7]:
print(dataset['City_encoded'].unique()) # Just for checking whether my cities were encoded properly or not
print(dataset['City'].unique())


KeyError: 'City_encoded'

**Converting "Has Table booking" and "Has Online delivery" into binary.**

In [ ]:
dataset['Has Table booking'] = dataset['Has Table booking'].apply(lambda x: 1 if x == 'Yes' else 0)
dataset['Has Online delivery'] = dataset['Has Online delivery'].apply(lambda x: 1 if x == 'Yes' else 0)



*   When we will return the output we will convert it into Yes/NO.
*   Just for better way understanding for machine we have converted into Binary.



**From Here our encoding process will take place**

In [ ]:
#Importing LabelEncoder
from sklearn.preprocessing import LabelEncoder
city_encoder = LabelEncoder()
dataset['City_encoded'] = city_encoder.fit_transform(dataset['City'])



1.    LabelEncoder is used to convert categorical text data into numerical labels.
2.   fit: It learns the unique categories present in the City column (e.g., "Mumbai", "New Delhi", "Bangalore", etc).
3.   transform: It converts each unique city name into a numerical value (integer) based on its label.
For eg Mumbai City - 0
       Delhi City - 1 and so on..

**Now we will explore TfdifVectorizer tool.**

1.   The TfidfVectorizer in Python is a powerful tool provided by the scikit-learn library to convert a collection of text documents into numerical feature vectors.
2.   Tokenization: Basically it breaks down the text into individual terms (words).



In [ ]:
# Importing TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Using TF-IDF for Cuisines
tfidf_vectorizer = TfidfVectorizer()
cuisine_matrix = tfidf_vectorizer.fit_transform(dataset['Cuisines'])

**Now we will combine other necessary features into a single Dataframe.**

In [ ]:
features = pd.concat([
    pd.DataFrame(cuisine_matrix.toarray()),  # Cuisines as TF-IDF features
    dataset[['Average Cost for two', 'Aggregate rating', 'Has Table booking', 'Has Online delivery', 'City_encoded']]
], axis=1)

**Selecting our algorithm**

In [ ]:
#Importing our algorithm
from sklearn.neighbors import NearestNeighbors

**I have decided to use K-Nearest Neighbors(KNN) algorithm for both classification and regression tasks**

**First we will convert all column names to strings to avoid TypeError**

In [ ]:
features.columns = features.columns.astype(str)

In [ ]:
knn_model = NearestNeighbors(n_neighbors=5, algorithm='auto')
knn_model.fit(features) #we will simply fit our features into the model

NearestNeighbors()

**Here's how KNN algorithm will help me for Restaurant Recommendation System!**

1.   The goal of our recommendation system is to suggest restaurants that match a user's preferences (e.g., cuisine type, cost, rating, location, etc).
2.   The KNN model will identify the most similar restaurants to a user's input based on the features I have defined (like cuisine, cost, rating, etc).
3.   It works by calculating the distance between the user's preference vector and all the restaurants in your dataset.



**Now we will define a user's preference function.**

In [ ]:
def recommend_restaurants(cuisine, cost, rating, city, table_booking=False, online_delivery=False, n_recommendations=5):
    # It handles the city input
    if city not in city_encoder.classes_:
        print(f"City '{city}' not found. Using most common city.")
        city_encoded = city_encoder.transform([dataset['City'].mode()[0]])[0]
    else:
        city_encoded = city_encoder.transform([city])[0]

    # As we have mentioned before we use TF-IDF to transform the input cuisine.
    user_cuisine_vector = tfidf_vectorizer.transform([cuisine]).toarray()

    table_booking_val = 1 if table_booking else 0
    online_delivery_val = 1 if online_delivery else 0

    # Basically it creates user preference as a DataFrame with the same column names.
    user_preference = np.hstack(( # hstack is a horrizontal stack that concatenate two arrays together
        user_cuisine_vector[0],
        [cost, rating, table_booking_val, online_delivery_val, city_encoded]
    )).reshape(1, -1)

    #We are converting to DataFrame with the same column names as the training data.
    user_preference_df = pd.DataFrame(user_preference, columns=features.columns)

    # It finds the nearest neighbors.
    distances, indices = knn_model.kneighbors(user_preference_df, n_neighbors=n_recommendations)

    # It basically retrieve recommended restaurants.
    recommendations = dataset.iloc[indices[0]]

    #As we have mentioned before we will convert binary values to 'Yes'/'No' for better readability
    recommendations = dataset.iloc[indices[0]].copy() # Just to avoid future errors
    recommendations['Has Table booking'] = recommendations['Has Table booking'].replace({0: 'No', 1: 'Yes'})
    recommendations['Has Online delivery'] = recommendations['Has Online delivery'].replace({0: 'No', 1: 'Yes'})

    return recommendations[['Restaurant Name', 'Cuisines', 'City', 'Address', 'Average Cost for two',
                            'Aggregate rating', 'Has Table booking', 'Has Online delivery']]

# It's basically an example for user preferences.
recommendations = recommend_restaurants(
    cuisine='Japanese',
    cost=1500,
    rating=4.5,
    city='Mumbai',
    table_booking=True,
    online_delivery=False
)

print(recommendations)

                         Restaurant Name  \
2494  The English Department Bar & Diner   
2489                          Farzi Cafe   
2484                      145 Kala Ghoda   
2495                     Mirchi And Mime   
2490                           SpiceKlub   

                                               Cuisines    City  \
2494  Italian, Continental, Mexican, Japanese, Ameri...  Mumbai   
2489                                      Modern Indian  Mumbai   
2484                     Fast Food, Beverages, Desserts  Mumbai   
2495                North Indian, South Indian, Mughlai  Mumbai   
2490                                       North Indian  Mumbai   

                                                Address  Average Cost for two  \
2494  Malad link Road, Near Inorbit Mall Junction, M...                  1500   
2489  Kamala Mills, Near Radio Mirchi Office, Lower ...                  1500   
2484                      145, Kala Ghoda, Fort, Mumbai                  1500   
2495  Tr